### CArga de bases


In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from variables_inicio import *
from sqlalchemy import create_engine
from sqlalchemy import text

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()



In [2]:
fecha_mes_base='2026-07-01'

fecha_envio_base='2026-07-27'
# filename='TARGET_202606_05.xlsx'
filename='TARGET_202607_F.xlsx'

ruta_archivo = os.path.join(ruta_csv, filename)
df_recepcion = pd.read_excel(ruta_archivo)
print(df_recepcion.columns.tolist())
print(df_recepcion.shape)


['ID PROVEEDOR', 'NOMBRES', 'APELLIDO PATERNO', 'APELLIDO MATERNO', 'TIPO DE DOCUMENTO', 'NUMERO_DOCUMENTO', 'TASA(TEA)', 'TEA_DF', 'TELEFONO_CASA1', 'TELEFONO_CASA2', 'TELEFONO_CASA3', 'TELEFONO_CASA4', 'TELEFONO_CASA5', 'TELEFONO_CASA6', 'TELEFONO_CASA7', 'TELEFONO_CASA8', 'TELEFONO_CASA9', 'TELEFONO_CASA10', 'NUEVO_GRUPO03', 'PROB_CONTACTO', 'DINERS_NRO1', 'DINERS_NRO2', 'DINERS_NRO3', 'DINERS_NRO4', 'DINERS_NRO5', 'OSIPTEL_NRO1_ENCRIP', 'OSIPTEL_NRO2_ENCRIP', 'OSIPTEL_NRO3_ENCRIP', 'OSIPTEL_NRO4_ENCRIP', 'OSIPTEL_NRO5_ENCRIP', 'DCP01', 'DCP02', 'DCP03', 'DCP04', 'DCP05', 'TELEFONO_1', 'TELEFONO_2', 'TELEFONO_3', 'CELULAR04', 'CELULAR05', 'CELULAR06', 'CELULAR07', 'CELULAR08', 'CELULAR09', 'CELULAR10', 'RECENCIA', 'PERFIL', 'PRODUCTO', 'LINEA_DIN', 'ID RESULTADO GESTION', 'TCEA_NEW', 'TCEA_CUOTAS_DF', 'LIMACALLAO', 'FECHA_NACIMIENTO', 'PROVINCIA', 'L_BCP', 'L_BBVA', 'L_IBK', 'L_SCO', 'L_BIF', 'L_CITI', 'L_FIN', 'L_RIP', 'L_CMR', 'L_CRESCO', 'L_CEN', 'L_AZT', 'L_UNO', 'L_GNB', 'L_EFE

In [3]:
df_recepcion['NUMERO_DOCUMENTO'] = df_recepcion['NUMERO_DOCUMENTO'].astype(str).str.zfill(8)

print('Total recibido:',df_recepcion.shape[0])
print('Total recibido sin duplicados:',df_recepcion['NUMERO_DOCUMENTO'].nunique())

ruta_archivo = os.path.join(ruta_csv, 'recepcion_mes_tc.csv')
df_recepcion.to_csv(ruta_archivo,index=False,sep=';',encoding='utf-8-sig')

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_kishin}:{pwd_kishin}@{server_kishin}/{db_kishin}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
    SELECT
        NUMERO_DOCUMENTO
    FROM DANTALION.dbo.Base_Maestra_Diners_TC_Vigente
    WHERE TRY_CONVERT(DATE, fecha_envio) < cast('{fecha_envio_base}' as date)
"""
df_vigente = pd.read_sql(query, engine_kishin)

Total recibido: 36911
Total recibido sin duplicados: 36911


In [4]:
dni_vigente = set(df_vigente['NUMERO_DOCUMENTO'].dropna().astype(str))
dni_recepcion = set(df_recepcion['NUMERO_DOCUMENTO'].dropna().astype(str))

In [5]:
dni_nuevos = dni_recepcion - dni_vigente
print("Lead nuevos:", len(dni_nuevos))


Lead nuevos: 22960


In [6]:
dni_comunes = dni_vigente & dni_recepcion
print("Lead en comun:", len(dni_comunes))


Lead en comun: 13951


In [7]:
dni_retirados = dni_vigente - dni_recepcion
print("Lead retirados:", len(dni_retirados))


Lead retirados: 26201


In [8]:
print("Lead recepcion:", len(dni_recepcion))
print("Lead vigente:", len(dni_vigente))


Lead recepcion: 36911
Lead vigente: 40152


In [9]:

dni_retirados = dni_vigente - dni_recepcion
print("Lead retirados:", len(dni_retirados))
dni_nuevos = dni_recepcion - dni_vigente
print("Lead nuevos:", len(dni_nuevos))


dni_nuevos = dni_recepcion - dni_vigente
if dni_nuevos:
    df_nuevos = pd.DataFrame({
        'NUMERO_DOCUMENTO': sorted(dni_nuevos)
    })

    ruta_archivo = os.path.join(ruta_csv, 'recepcion_nuevo_tc.csv')
    df_nuevos.to_csv(ruta_archivo,index=False,encoding='utf-8-sig')
    print('se guardo registros nuevos')
else:
    print('no se tienen regisros nuevos')

dni_comunes = dni_vigente & dni_recepcion
if dni_comunes:
    df_nuevos = pd.DataFrame({
        'NUMERO_DOCUMENTO': sorted(dni_comunes)
    })

    ruta_archivo = os.path.join(ruta_csv, 'recepcion_comun_tc.csv')
    df_nuevos.to_csv(ruta_archivo,index=False,encoding='utf-8-sig')
    print('se guardo regisros en comun')
else:
    print('no se tienen regisros en comun')


Lead retirados: 26201
Lead nuevos: 22960
se guardo registros nuevos
se guardo regisros en comun


In [ ]:
SI NO SE TIENE REGISTROS ENCOMUN PUES EL ARCHIVO COMUN DE ESTAR VACIO

In [10]:
query = f"""
    SELECT top(0)*
    FROM [DANTALION].[dbo].[Base_Maestra_Diners_TC]
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

filename='recepcion_mes_tc.csv'
df_base=cargar_archivo_csv(spark,filename,';',True)

# filename='recepcion_comun_tc.csv'
# df_comun=cargar_archivo_csv(spark,filename,';',True)
# overwrite_table_SQL(spark,df_comun,f'update_reg_dup_env_diners_tc',server_kishin,user_kishin,pwd_kishin,'DANTALION')

print(df_base.count())
# print(df_comun.count())

36911


In [11]:
fecha_mes_base

'2026-07-01'

In [ ]:
CHANCAR INFO DE ARCHIVO

In [6]:
from sqlalchemy import text
try:
    with engine_kishin.begin() as conn:

        query_1 = f"""
            DECLARE @fecha_inicio DATE = TRY_CONVERT(DATE, '{fecha_mes_base}');
            DECLARE @fecha_fin DATE = DATEADD(MONTH, 1, @fecha_inicio);

            delete A
            FROM DANTALION.dbo.Base_Maestra_Diners_TC A
            INNER JOIN DANTALION.dbo.borrar_base_actual_diners_tc B
                ON A.NUMERO_DOCUMENTO = B.NUMERO_DOCUMENTO
            WHERE TRY_CONVERT(DATE, A.fecha_envio,23) >= @fecha_inicio
            AND TRY_CONVERT(DATE, A.fecha_envio,23) < @fecha_fin
            """

        result_1 = conn.execute(text(query_1))
        print("filas eliminadas:", result_1.rowcount)

        query_2 = f"""
            DECLARE @fecha_envio_ref DATE = TRY_CONVERT(DATE, '{fecha_envio_base}');

            UPDATE DANTALION.dbo.Base_Maestra_Diners_TC
            SET RETIRO = 'RETIRO_ENVIO'
            WHERE TRY_CONVERT(DATE, fecha_envio,23) = @fecha_envio_ref 
            """

        result_2 = conn.execute(text(query_2))
        print("en retiro:", result_2.rowcount)

except Exception as e:
    print("Error:", e)

filas eliminadas: 15547
en retiro: 0


In [12]:
df_base = df_base.withColumn(
        "NUMERO_DOCUMENTO",
        F.lpad(F.col("NUMERO_DOCUMENTO").cast("string"), 8, "0")
    )

In [13]:
exprs = [
    F.count(F.when(F.col(c).isNotNull(), c)).alias(c)
    for c in df_base.columns
]

df_counts = df_base.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base = df_base.select(cols_con_data)

In [14]:
df_base=df_base.withColumnRenamed('LINEA_DIN','LINEA CREDITO DOLARES')
df_base=df_base.withColumnRenamed('CELULAR01','CELULAR1')
df_base=df_base.withColumnRenamed('CELULAR02','CELULAR2')
df_base=df_base.withColumnRenamed('CELULAR03','CELULAR3')
df_base=df_base.withColumnRenamed('CELULAR04','CELULAR4')
df_base=df_base.withColumnRenamed('RECENCIA','RECURRENCIA')
df_base=df_base.withColumnRenamed('FECHA_NACIMIENTO','FEC_NAC')
df_base=df_base.withColumnRenamed('NUEVO_GRUPO03','PROB_CONTACTO')
df_base=df_base.withColumnRenamed('EX_SOCIO','MARCA')
df_base=df_base.withColumnRenamed('FUENTE','N_BASE')

df_base = df_base.withColumn(
    "LIMAPROVINCIA",
    F.when(F.col("PROVINCIA")=='LIMA', F.lit("LIMA"))
    .otherwise(F.lit('PROVINCIA'))
)


In [22]:
df_base=df_base.drop( 'LIMACALLAO','TELEFONO_1','TELEFONO_2','TELEFONO_3')


In [ ]:
['ID PROVEEDOR', 'NOMBRES', 'APELLIDO PATERNO', 'TIPO DE DOCUMENTO', 'NUMERO_DOCUMENTO', 'TASA(TEA)', 'TEA_DF', 'PROB_CONTACTO', 'TELEFONO_1', 'TELEFONO_2', 'TELEFONO_3', 'RECURRENCIA', 'PERFIL', 'PRODUCTO', 'LINEA CREDITO DOLARES', 'ID RESULTADO GESTION', 'TCEA', 'TCEA_CUOTAS_DF', 'FEC_NAC', 'PROVINCIA', 'L_BCP', 'L_BBVA', 'L_IBK', 'L_SCO', 'L_BIF', 'L_CITI', 'L_FIN', 'L_RIP', 'L_CMR', 'L_CRESCO', 'L_CEN', 'L_AZT', 'L_UNO', 'L_GNB', 'L_EFE', 'L_COM', 'L_NAC', 'MARCA', 'LINEA_ANT_EX_SOCIO', 'ULT_TASA_EX_SOCIO', 'LIMAPROVINCIA', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO']

['ID PROVEEDOR', 'NOMBRES', 'APELLIDO PATERNO', 'TIPO DE DOCUMENTO', 'NUMERO_DOCUMENTO', 'TASA(TEA)', 'TEA_DF', 'PROB_CONTACTO', 'TELEFONO_1', 'TELEFONO_2', 'TELEFONO_3', 'RECURRENCIA', 'PERFIL', 'PRODUCTO', 'LINEA CREDITO DOLARES', 'ID RESULTADO GESTION', 'TCEA', 'TCEA_CUOTAS_DF', 'FEC_NAC', 'PROVINCIA', 'L_BCP', 'L_BBVA', 'L_IBK', 'L_SCO', 'L_BIF', 'L_CITI', 'L_FIN', 'L_RIP', 'L_CMR', 'L_CRESCO', 'L_CEN', 'L_AZT', 'L_UNO', 'L_GNB', 'L_EFE', 'L_COM', 'L_NAC', 'MARCA', 'LINEA_ANT_EX_SOCIO', 'ULT_TASA_EX_SOCIO', 'LIMAPROVINCIA', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO']


In [21]:

df_base=df_base.withColumn('AÑO_DURACION_BASE',F.lit('2026'))
df_base=df_base.withColumn('MES_DURACION_BASE',F.lit('07'))
df_base=df_base.withColumnRenamed('TCEA_NEW','TCEA')
df_base=df_base.withColumn('FECHA_ENVIO',F.lit('2026-07-27'))
df_base=df_base.withColumn('SERVICIO',F.lit('02'))



df_base = df_base.withColumn(
    "CEL01",
    F.when(F.col("TELEFONO_1").rlike(r"^9\d{0,8}$"), F.col("TELEFONO_1"))
)
df_base = df_base.withColumn(
    "CEL02",
    F.when(F.col("TELEFONO_2").rlike(r"^9\d{0,8}$"), F.col("TELEFONO_2"))
)
df_base = df_base.withColumn(
    "CEL03",
    F.when(F.col("TELEFONO_3").rlike(r"^9\d{0,8}$"), F.col("TELEFONO_3"))
)
# df_base = df_base.withColumn(
#     "CEL04",
#     F.when(F.col("CELULAR4").rlike(r"^9\d{0,8}$"), F.col("CELULAR4"))
# )


In [23]:
df_base.show(2)

+------------+-------+----------------+-----------------+----------------+---------+-------+-------------+-----------+---------+--------+---------------------+--------------------+-----+--------------+----------+---------+-----+------+-----+-----+-----+------+-----+-----+-----+--------+-----+-----+-----+-----+-----+-----+-----+-----+------------------+-----------------+-------------+-----------------+-----------------+-----------+--------+---------+-----+-----+
|ID PROVEEDOR|NOMBRES|APELLIDO PATERNO|TIPO DE DOCUMENTO|NUMERO_DOCUMENTO|TASA(TEA)| TEA_DF|PROB_CONTACTO|RECURRENCIA|   PERFIL|PRODUCTO|LINEA CREDITO DOLARES|ID RESULTADO GESTION| TCEA|TCEA_CUOTAS_DF|   FEC_NAC|PROVINCIA|L_BCP|L_BBVA|L_IBK|L_SCO|L_BIF|L_CITI|L_FIN|L_RIP|L_CMR|L_CRESCO|L_CEN|L_AZT|L_UNO|L_GNB|L_EFE|L_COM|L_NAC|MARCA|LINEA_ANT_EX_SOCIO|ULT_TASA_EX_SOCIO|LIMAPROVINCIA|AÑO_DURACION_BASE|MES_DURACION_BASE|FECHA_ENVIO|SERVICIO|    CEL01|CEL02|CEL03|
+------------+-------+----------------+-----------------+-----------

In [ ]:
# df_base=df_base.withColumn('FECHA_ENVIO',F.lit('2026-07-01'))


In [23]:
df_base = (
    df_base
    .withColumn(
        "CELULAR3",
        F.regexp_replace(
            F.col("CELULAR3").cast("string"),
            r"\.0$",
            ""
        )
    )
    .withColumn(
        "CEL03",
        F.when(
            F.col("CELULAR3").rlike(r"^9\d{8}$"),
            F.col("CELULAR3")
        )
    )
)

In [24]:
df_base=df_base.withColumnRenamed('ULT_TASA_EX_SOCIO','TASA_ANT')
df_base=df_base.withColumnRenamed('LINEA_ANT_EX_SOCIO','MAYOR_LINEA')


In [25]:

cols_base = set(df_base.columns)
cols_formato = set(df_formato.columns)

solo_en_base = cols_base - cols_formato
print("Solo en df_base:", solo_en_base)

Solo en df_base: set()


In [26]:
print(df_base.count())
print(df_base.dropDuplicates(['NUMERO_DOCUMENTO']).count())


36911
36911


In [27]:
df_base.groupBy('PROB_CONTACTO') \
    .count() \
    .orderBy('PROB_CONTACTO') \
    .show(30)


+-------------+-----+
|PROB_CONTACTO|count|
+-------------+-----+
|            A|  134|
|            B|  322|
|            C|  203|
|            D|  214|
|            E|    7|
|            F| 7367|
|            G|15723|
|            H|  635|
+-------------+-----+



In [44]:
df_base.show(2)

+------------+-------+----------------+-----------------+----------------+---------+-------+-------------+---------+--------+--------+--------+-----------+---------+--------+---------------------+--------------------+-----+--------------+----------+---------+-----+------+-----+-----+-----+------+-----+-----+-----+--------+-----+-----+-----+-----+-----+-----+-----+-----+-----------+--------+-------------+-----------------+-----------------+-----------+--------+---------+-----+-----+-----+
|ID PROVEEDOR|NOMBRES|APELLIDO PATERNO|TIPO DE DOCUMENTO|NUMERO_DOCUMENTO|TASA(TEA)| TEA_DF|PROB_CONTACTO| CELULAR1|CELULAR2|CELULAR3|CELULAR4|RECURRENCIA|   PERFIL|PRODUCTO|LINEA CREDITO DOLARES|ID RESULTADO GESTION| TCEA|TCEA_CUOTAS_DF|   FEC_NAC|PROVINCIA|L_BCP|L_BBVA|L_IBK|L_SCO|L_BIF|L_CITI|L_FIN|L_RIP|L_CMR|L_CRESCO|L_CEN|L_AZT|L_UNO|L_GNB|L_EFE|L_COM|L_NAC|MARCA|MAYOR_LINEA|TASA_ANT|LIMAPROVINCIA|AÑO_DURACION_BASE|MES_DURACION_BASE|FECHA_ENVIO|SERVICIO|    CEL01|CEL02|CEL03|CEL04|
+------------+

In [27]:
append_table_SQL(spark,df_base,'Base_Maestra_Diners_TC',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [28]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners_Tc", "SP tNumeros diners_tc")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners_Tc", "SP actualizar diners_tc Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners_Tc", "SP actualizar diners_tc SA")

SP tNumeros diners_tc | realizado | duración: 321.75 seg
SP actualizar diners_tc Zeus | realizado | duración: 360.16 seg
SP actualizar diners_tc SA | realizado | duración: 6.19 seg


In [ ]:
query = f"""
    SELECT  [ID PROVEEDOR]
      ,[NOMBRES]
      ,[APELLIDO PATERNO]
      ,[APELLIDO MATERNO]
      ,[TIPO DE DOCUMENTO]
      ,[NUMERO_DOCUMENTO]
      ,[TASA(TEA)]
      ,[TEA_DF]
      ,[TELEFONO_CASA1]
      ,[TELEFONO_CASA2]
      ,[TELEFONO_CASA3]
      ,[TELEFONO_CASA4]
      ,[TELEFONO_CASA5]
      ,[TELEFONO_CASA6]
      ,[TELEFONO_CASA7]
      ,[TELEFONO_CASA8]
      ,[TELEFONO_CASA9]
      ,[TELEFONO_CASA10]
      ,[CELULAR1]
      ,[CELULAR2]
      ,[CELULAR3]
      ,[CELULAR4]
      ,[CELULAR5]
      ,[CELULAR6]
      ,[CELULAR7]
      ,[CELULAR8]
      ,[CELULAR9]
      ,[CELULAR10]
      ,[PERFIL]
      ,[PRODUCTO]
      ,[LINEA CREDITO DOLARES]
      ,[ID RESULTADO GESTION]
      ,[TCEA]
      ,[TCEA_CUOTAS_DF]
      ,[LIMAPROVINCIA]
      ,[GRUPO_EJECUCION]
      ,[FEC_NAC]
      ,[PROVINCIA]
      ,[L_BCP]
      ,[L_BBVA]
      ,[L_IBK]
      ,[L_SCO]
      ,[L_BIF]
      ,[L_CITI]
      ,[L_FIN]
      ,[L_RIP]
      ,[L_CMR]
      ,[L_CRESCO]
      ,[L_CEN]
      ,[L_AZT]
      ,[L_UNO]
      ,[L_GNB]
      ,[L_EFE]
      ,[L_COM]
      ,[L_NAC]
      ,[RECURRENCIA]
      ,[CEL01]
      ,[CEL02]
      ,[CEL03]
      ,[CEL04]
      ,[CEL05]
      ,[CEL06]
      ,[CEL07]
      ,[CEL08]
      ,[CEL09]
      ,[CEL10]
      ,[TELF1]
      ,[TELF2]
      ,[TELF3]
      ,[TELF4]
      ,[TELF5]
      ,[TELF6]
      ,[TELF7]
      ,[TELF8]
      ,[FECHA_ENVIO]
      ,[MES_DURACION_BASE]
      ,[AÑO_DURACION_BASE]
      ,[SERVICIO]
      ,[RETIRO]
      ,[MARCA]
      ,[MARCA2]
      ,[FLAT1]
      ,[FLAT2]
      ,[FLAT3]
      ,[REP1]
      ,[REP2]
      ,[TASA_ANT]
      ,[TCEA_ANT]
      ,[FLG_BT]
      ,[MONTO_OFERTA_BT]
      ,[PRIORIDAD_TELEFONO_1]
      ,[PRIORIDAD_TELEFONO_2]
      ,[PRIORIDAD_TELEFONO_3]
      ,[PRIORIDAD_TELEFONO_4]
      ,[PRIORIDAD_TELEFONO_5]
      ,'Base util' as [Estado_base]
      , '202605'as [periodo]
  FROM DANTALION.[dbo].Base_Maestra_Diners_TC
  where FECHA_ENVIO='2026-05-13'
  and NUMERO_DOCUMENTO is not null
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)


In [ ]:
hugo.lopez@targetoutsourcing.com.pe

In [ ]:
from pyspark.sql import functions as F



In [ ]:
d

Solo en df_base: {'EX_SOCIO', 'NUEVO_GRUPO03', 'LIMACALLAO'}


In [ ]:
df = df_vigente.merge(df_tc, on='NUMERO_DOCUMENTO', how='inner')
list_dni = (
    df['NUMERO_DOCUMENTO']
    .dropna()
    .drop_duplicates()
    .tolist()
)

In [ ]:
df_prueba=df_prueba.drop( 'ULT_TASA_EX_SOCIO', 'LIMACALLAO', 'LINEA_ANT_EX_SOCIO')


In [2]:
df.count()

NUMERO_DOCUMENTO    1833
dtype: int64

In [2]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners_TC
            SET CASHBACK = 'Cash back 300'
            WHERE NUMERO_DOCUMENTO IN ({in_clause})
                and TRY_CONVERT(DATE, fecha_envio) >= TRY_CONVERT(DATE, '{fecha_mes_base}')
                AND TRY_CONVERT(DATE, fecha_envio) < DATEADD(MONTH, 1, TRY_CONVERT(DATE, '{fecha_mes_base}'))
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 1771


In [3]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners_tc", "SP tNumeros diners TC")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC SA")

SP tNumeros diners TC | realizado | duración: 206.37 seg
SP actualizar diners TC Zeus | realizado | duración: 62.1 seg
SP actualizar diners TC SA | realizado | duración: 2.45 seg


In [ ]:


server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
fecha_mes_base='2026-06-01'
campana=fecha_a_nombre('2026-05-01')
query = f"""
select dni as NumDoc, 1 as venta_target from SAMANTHA.dbo.Ventas_Target
where CAMPANA='Diners'
and CONVERT(DATE, FECHA) >= CONVERT(DATE, '{fecha_mes_base}')
AND CONVERT(DATE, FECHA) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
   
"""
df_venta = pd.read_sql(query, engine_zeus)

df_target = df_tc.merge(
    df_venta,
    on='NumDoc',
    how='left'
)
df_final = df_target.merge(
    df,
    on='NumDoc',
    how='inner'
)
df_final = (
    df_final[df_final['venta_target'].isnull()]
    .drop(columns=['venta_target'])
)
df_final.rename(
    columns={
        'Importe Solicitado': 'Monto'
    },
    inplace=True
)
print(df_final.columns.tolist())

c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


['NumDoc', 'TIPO_PRODUCTO', 'RETIRO', 'Canal', 'Autor', 'Subcanal', 'Motivo', 'Monto', 'fecha', 'hora']


In [72]:
df_final=df_final[df_final['Monto'].notnull()]

In [78]:
df_final[['TIPO_PRODUCTO','Canal', 'Monto', 'fecha']].head()


,TIPO_PRODUCTO,Canal,Monto,fecha
0,PPD,CANALES DIGITALES,41100.0,2026-06-04
1,PPD,CANALES DIGITALES,13800.0,2026-06-04
2,PPD,CONTACT CENTER,6000.0,2026-06-04


In [ ]:

total_monto_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].sum()

total_ope_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].count()

print(f'PPD | Monto total: {int(total_monto_ppd)} |  Total operaciones {int(total_ope_ppd)} ')

PPD | Monto total: 60900 |  Total operaciones 3 


In [ ]:
# ruta_archivo = os.path.join(ruta_csv, 'PPD_plus.xlsx')
# df_list_filtrada.to_excel(ruta_archivo, index=False)

In [ ]:
list_dni = (
    df_final.loc[df_final['RETIRO'].isna(), 'NumDoc']
    .dropna()
    .drop_duplicates()
    .tolist()
)

in_clause = ",".join(f"'{x}'" for x in list_dni)

try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners
            SET RETIRO = 'RETIRO'
            WHERE NumDoc IN ({in_clause})
                and CONVERT(DATE, fecha_envio) >= CONVERT(DATE, '{fecha_mes_base}')
                AND CONVERT(DATE, fecha_envio) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)


In [77]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners", "SP tNumeros diners")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners", "SP actualizar diners Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners", "SP actualizar diners SA")

SP tNumeros diners | realizado | duración: 5.48 seg
SP actualizar diners Zeus | realizado | duración: 5.54 seg
SP actualizar diners SA | realizado | duración: 1.5 seg
